# Team aEC/T Leaderboard

Rank UFA teams by possession aEC per throw. The main rate is `team_aec_per_throw = sum(total_aec) / sum(throw_count)`, which treats every throw as one unit in the team rate.

In [18]:
import base64
import importlib
import mimetypes
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ufa_aec_possessions
import ufa_aec_possessions.fetching
import ufa_aec_possessions.leaderboards

importlib.reload(ufa_aec_possessions.fetching)
importlib.reload(ufa_aec_possessions.leaderboards)
importlib.reload(ufa_aec_possessions)

from ufa_aec_possessions import (
    build_possessions,
    build_scoring_possessions,
    fetch_shownspace_season_throws_cached,
    filter_analysis_possessions,
    summarize_team_aec_consistency,
    summarize_team_aec_per_throw,
    summarize_team_top_possessions,
)

In [ ]:
SEASON = 2026
MAX_GAMES = None
FORCE_REFRESH = False
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'shownspace'
MIN_POSSESSIONS = 20
TOP_TEAMS = 22
TOP_POSSESSION_COUNT = 5
TOP_POSSESSION_MIN_THROWS = 5
CONSISTENCY_TOP_N = 10
CONSISTENCY_MIN_THROWS = 5

## Load League Possessions

In [20]:
league_games, league_throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)
league_all_possessions, league_all_paths = build_possessions(league_throws)
league_possessions, league_paths = build_scoring_possessions(league_throws)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'League all possessions found: {len(league_all_possessions):,}')
print(f'League scoring possessions found: {len(league_possessions):,}')

League games loaded: 132
League throws loaded: 72,616
League all possessions found: 10,331
League scoring possessions found: 5,259


In [21]:
LEADERBOARD_COLUMNS = [
    'rank', 'team_id', 'team_aec_per_throw', 'avg_possession_aec_per_throw',
    'possessions', 'games', 'throws', 'total_aec', 'avg_throw_count',
    'avg_field_progress', 'huck_rate', 'resets_per_possession', 'o_line_share',
]

FORMATTERS = {
    'team_aec_per_throw': '{:.3f}',
    'avg_possession_aec_per_throw': '{:.3f}',
    'median_possession_aec_per_throw': '{:.3f}',
    'total_aec': '{:.1f}',
    'avg_throw_count': '{:.1f}',
    'avg_field_progress': '{:.1f}',
    'huck_rate': '{:.1%}',
    'resets_per_possession': '{:.1f}',
    'o_line_share': '{:.1%}',
    'top_mean_metric': '{:.3f}',
    'top_median_metric': '{:.3f}',
    'top_floor_metric': '{:.3f}',
    'top_ceiling_metric': '{:.3f}',
    'top_total_aec': '{:.1f}',
    'top_avg_throw_count': '{:.1f}',
    'median_metric': '{:.3f}',
    'p75_metric': '{:.3f}',
    'p90_metric': '{:.3f}',
    'top_n_mean_metric': '{:.3f}',
    'high_metric_share': '{:.1%}',
    'high_metric_threshold': '{:.3f}',
    'goal_share': '{:.1%}',
    'turnover_share': '{:.1%}',
}

TOP_POSSESSION_COLUMNS = [
    'rank', 'team_id', 'top_mean_metric', 'top_floor_metric', 'top_avg_throw_count',
    'team_aec_per_throw', 'median_possession_aec_per_throw', 'possessions', 'throws',
]

CONSISTENCY_COLUMNS = [
    'rank', 'team_id', 'median_metric', 'p75_metric', 'top_n_mean_metric',
    'high_metric_share', 'team_aec_per_throw', 'possessions', 'throws', 'avg_throw_count',
    'goal_share', 'turnover_share',
]


def show_team_aec_t_table(leaderboard, n=TOP_TEAMS, columns=LEADERBOARD_COLUMNS):
    table = leaderboard.head(n).reindex(columns=columns).copy()
    for column, formatter in FORMATTERS.items():
        if column in table:
            values = pd.to_numeric(table[column], errors='coerce')
            table[column] = values.map(lambda value: '' if pd.isna(value) else formatter.format(value))
    return table


POSITIVE_COLOR_SCALE = [
    [0.0, '#eaf4e8'],
    [0.35, '#9fd391'],
    [0.70, '#3f9c5a'],
    [1.0, '#09552f'],
]
DIVERGING_COLOR_SCALE = [
    [0.0, '#ba4a3c'],
    [0.50, '#f4f7f2'],
    [1.0, '#0b5d3b'],
]
HOVER_FORMATS = {
    'team_aec_per_throw': ':.3f',
    'avg_possession_aec_per_throw': ':.3f',
    'median_possession_aec_per_throw': ':.3f',
    'top_mean_metric': ':.3f',
    'top_floor_metric': ':.3f',
    'median_metric': ':.3f',
    'p75_metric': ':.3f',
    'top_n_mean_metric': ':.3f',
    'high_metric_share': ':.1%',
    'possessions': ':,',
    'games': ':,',
    'throws': ':,',
    'huck_rate': ':.1%',
    'o_line_share': ':.1%',
    'goal_share': ':.1%',
    'turnover_share': ':.1%',
}
HOVER_LABELS = {
    'team_aec_per_throw': 'team aEC/T',
    'avg_possession_aec_per_throw': 'avg possession aEC/T',
    'median_possession_aec_per_throw': 'median possession aEC/T',
    'top_mean_metric': 'top mean',
    'top_floor_metric': 'top floor',
    'median_metric': 'median',
    'p75_metric': '75th percentile',
    'top_n_mean_metric': 'top-N mean',
    'high_metric_share': 'high-rate share',
    'possessions': 'possessions',
    'games': 'games',
    'throws': 'throws',
    'huck_rate': 'huck rate',
    'o_line_share': 'O-line share',
    'goal_share': 'goal share',
    'turnover_share': 'turnover share',
}
TEAM_LOGOS = {
    'alleycats': 'https://watchufa.com/sites/default/files/IND-Logo-Teams.png',
    'apex': 'https://watchufa.com/sites/default/files/Apex-HeaderLogo.png',
    'bighorns': 'https://watchufa.com/sites/default/files/logo-team-bighorns.png',
    'breeze': 'https://watchufa.com/sites/default/files/logo-team-DC.png',
    'cascades': 'https://watchufa.com/sites/default/files/logo-team-SEA.png',
    'empire': 'https://watchufa.com/sites/default/files/logo-team-NY.png',
    'flyers': 'https://watchufa.com/sites/default/files/logo-team-RAL_0.png',
    'glory': 'https://watchufa.com/sites/default/files/medium_logo-team-BOS.png',
    'growlers': 'https://watchufa.com/sites/default/files/logo-team-SD_0.png',
    'havoc': 'https://watchufa.com/sites/default/files/medium_logo-team_HTX.png',
    'hustle': 'https://watchufa.com/sites/default/files/medium_logo-team-ATL.png',
    'phoenix': 'https://watchufa.com/sites/default/files/logo-team-PHI.png',
    'radicals': 'https://watchufa.com/sites/default/files/logo-team-MAD.png',
    'royal': 'https://watchufa.com/sites/default/files/logo-team-MTL.png',
    'rush': 'https://watchufa.com/sites/default/files/logo-team-TOR_0.png',
    'shred': 'https://watchufa.com/sites/default/files/ShredWeb.png',
    'sol': 'https://watchufa.com/sites/default/files/logo-team-AUS_0.png',
    'spiders': 'https://watchufa.com/themes/AUDL_theme/css/images/logos/logo-team-SJ.png',
    'steel': 'https://watchufa.com/sites/default/files/Org_team_page.png',
    'thunderbirds': 'https://watchufa.com/sites/default/files/logo-team-PIT.png',
    'union': 'https://watchufa.com/sites/default/files/logo-team-CHI_0.png',
    'windchill': 'https://watchufa.com/sites/default/files/logo-team-MIN.png',
}
LOGO_SOURCE_CACHE = {}
TEAM_LOGO_CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'team_logos'
LOGO_MIME_TYPES = {
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
    '.png': 'image/png',
    '.svg': 'image/svg+xml',
    '.webp': 'image/webp',
}


def get_team_logo_source(team_id):
    team_id = str(team_id).lower()
    if team_id in LOGO_SOURCE_CACHE:
        return LOGO_SOURCE_CACHE[team_id]
    logo_url = TEAM_LOGOS.get(team_id)
    if not logo_url:
        return None
    suffix = Path(logo_url.split('?')[0]).suffix.lower() or '.png'
    cache_path = TEAM_LOGO_CACHE_DIR / f'{team_id}{suffix}'
    try:
        if not cache_path.exists():
            TEAM_LOGO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
            response = requests.get(logo_url, timeout=10)
            response.raise_for_status()
            cache_path.write_bytes(response.content)
        mime_type = LOGO_MIME_TYPES.get(cache_path.suffix.lower()) or mimetypes.guess_type(cache_path.name)[0] or 'image/png'
        logo_source = 'data:' + mime_type + ';base64,' + base64.b64encode(cache_path.read_bytes()).decode('ascii')
    except Exception:
        logo_source = logo_url
    LOGO_SOURCE_CACHE[team_id] = logo_source
    return logo_source


def plot_team_metric_leaderboard(leaderboard, metric, title, x_label, n=TOP_TEAMS):
    chart = leaderboard.head(n).sort_values(metric, ascending=True).copy()
    if chart.empty:
        return px.bar(title=title)
    chart['team_label'] = chart['rank'].astype(int).astype(str) + '. ' + chart['team_id'].astype(str).str.title()
    hover_columns = [
        column for column in [
            'team_aec_per_throw', 'avg_possession_aec_per_throw', 'median_possession_aec_per_throw',
            'top_mean_metric', 'top_floor_metric', 'median_metric', 'p75_metric',
            'top_n_mean_metric', 'high_metric_share', 'possessions', 'games', 'throws',
            'huck_rate', 'o_line_share', 'goal_share', 'turnover_share',
        ]
        if column in chart and column != metric
    ]
    metric_min = pd.to_numeric(chart[metric], errors='coerce').min()
    metric_max = pd.to_numeric(chart[metric], errors='coerce').max()
    has_negative_values = metric_min < 0
    crosses_zero = metric_min < 0 < metric_max
    color_scale = DIVERGING_COLOR_SCALE if has_negative_values else POSITIVE_COLOR_SCALE
    padding = max((metric_max - metric_min) * 0.10, 0.015)
    data_span = max(metric_max - min(metric_min, 0), 0.10)
    logo_space = max(data_span * 0.055, 0.010)
    x_range = [metric_min - padding - logo_space, metric_max + padding] if has_negative_values else [-logo_space, metric_max + padding]
    logo_x = x_range[0] + logo_space * 0.58
    logo_size_x = logo_space * 0.70
    hover_template = f'{x_label}=%{{x:.3f}}'
    for index, column in enumerate(hover_columns):
        label = HOVER_LABELS.get(column, column.replace('_', ' '))
        hover_template += f'<br>{label}=%{{customdata[{index}]{HOVER_FORMATS.get(column, "")}}}'
    hover_template += '<extra></extra>'
    fig = px.bar(
        chart,
        x=metric,
        y='team_label',
        orientation='h',
        text=metric,
        custom_data=hover_columns,
        labels={metric: x_label, 'team_label': 'Team'},
        color=metric,
        color_continuous_scale=color_scale,
    )
    fig.update_traces(
        texttemplate='%{x:.3f}',
        textposition='outside',
        cliponaxis=False,
        marker_line_color='rgba(255,255,255,0.85)',
        marker_line_width=0.8,
        opacity=0.96,
        hovertemplate=hover_template,
    )
    if crosses_zero:
        fig.add_vline(x=0, line_width=1, line_dash='dot', line_color='#6b7280')
        fig.update_coloraxes(cmid=0)
    for _, row in chart.iterrows():
        logo_source = get_team_logo_source(row['team_id'])
        if logo_source:
            fig.add_layout_image(
                dict(
                    source=logo_source,
                    xref='x',
                    yref='y',
                    x=logo_x,
                    y=row['team_label'],
                    sizex=logo_size_x,
                    sizey=0.72,
                    xanchor='center',
                    yanchor='middle',
                    layer='above',
                )
            )
    fig.update_layout(
        title={'text': title, 'x': 0.5, 'xanchor': 'center'},
        title_font={'size': 20, 'color': '#20385f'},
        height=max(560, 32 * len(chart) + 150),
        width=1040,
        margin={'l': 44, 'r': 78, 't': 78, 'b': 62},
        coloraxis_showscale=False,
        bargap=0.24,
        font={'family': 'Segoe UI, Arial, sans-serif', 'size': 12, 'color': '#23395d'},
        hoverlabel={
            'bgcolor': '#ffffff',
            'bordercolor': '#c9d8c4',
            'font': {'color': '#14213d', 'size': 12, 'family': 'Segoe UI, Arial, sans-serif'},
        },
        plot_bgcolor='#f7fbf5',
        paper_bgcolor='#ffffff',
    )
    fig.update_xaxes(
        title_text=x_label,
        range=x_range,
        showgrid=True,
        gridcolor='#e5eee2',
        zeroline=False,
        linecolor='#d7e4d2',
        tickfont={'color': '#31476b'},
        title_font={'color': '#31476b'},
    )
    fig.update_yaxes(
        title_text='',
        showgrid=False,
        showticklabels=True,
        tickmode='array',
        tickvals=chart['team_label'].tolist(),
        ticktext=(chart['rank'].astype(int).astype(str) + '.').tolist(),
        ticks='',
        tickfont={'color': '#20385f'},
    )
    return fig


def plot_team_aec_t_leaderboard(leaderboard, title, n=TOP_TEAMS):
    return plot_team_metric_leaderboard(
        leaderboard,
        metric='team_aec_per_throw',
        title=title,
        x_label='team aEC/T',
        n=n,
    )

## Metric Key

These charts use possession-level aEC, so the same team can look different depending on whether we summarize every throw, every possession, or only the team's best possessions.

- `team_aec_per_throw`: total team aEC divided by total throws in the selected possession pool. This is the team-level aEC/T rate.
- `avg_possession_aec_per_throw`: the average of each possession's aEC/T. This gives every possession equal weight, while `team_aec_per_throw` gives longer possessions more weight because they include more throws.
- `median_possession_aec_per_throw` / `median_metric`: the middle possession after sorting possessions by aEC/T. Half of the team's selected possessions are above this value and half are below it, so it is less affected by one huge point.
- `p75_metric`: the 75th percentile possession. About one quarter of the team's selected possessions are better than this. This is useful for asking, "how good are this team's good possessions, not just its absolute best one?"
- `p90_metric`: the 90th percentile possession. This is closer to a team's elite possessions, but still less extreme than only looking at the top five.
- `top_mean_metric`: the average metric value of the team's top selected possessions, usually the top five.
- `top_floor_metric`: the lowest value inside that top group. For a top-five chart, this is the fifth-best possession and tells us how strong the whole top five is.
- `top_n_mean_metric`: the average of a larger top group used for consistency checks. This rewards teams that repeatedly create high-aEC/T possessions instead of spiking once.
- `high_metric_share`: the share of selected possessions above the league-wide high-value threshold used in that consistency view.
- `huck_rate`: share of selected possessions with at least one huck. Several views exclude hucks so the chart focuses on worked, non-huck scoring possessions.
- `o_line_share`: share of selected possessions labeled as O-line possessions.
- `goal_share` / `turnover_share`: share of selected possessions ending in goals or turnovers. Turnover-inclusive views are usually better for judging overall team quality; scoring-only views are better for studying what successful possessions look like.


## Filter Note

The non-huck long-field scoring views are intentionally narrow: they look at possessions that score, start with meaningful field to travel, and do not rely on hucks. That makes them useful for studying possession shape, but they should not be read as a full team-quality ranking. The turnover-inclusive charts are the better check when the question is whether a team is actually efficient overall.

## All Possessions Including Turnovers

This is the closest view to Shown Space team `Tot-aEC`: it includes scoring possessions, turnovers, and other built possessions. This is the view where Bighorns show up as negative overall.

In [22]:
team_aec_t_all_possessions = summarize_team_aec_per_throw(
    league_all_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_possessions,
    title=f'{SEASON} UFA team aEC/T - all possessions including turnovers',
)

In [23]:
show_team_aec_t_table(team_aec_t_all_possessions)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,empire,0.077,0.115,453,12,3414,261.5,7.5,63.4,34.7%,2.0,58.1%
1,2,flyers,0.066,0.086,474,12,3467,227.4,7.3,58.9,36.5%,1.9,58.2%
2,3,windchill,0.065,0.088,441,12,3327,216.5,7.5,61.1,36.7%,2.1,62.4%
3,4,spiders,0.064,0.078,471,12,3394,215.7,7.2,55.3,24.0%,1.7,57.1%
4,5,breeze,0.061,0.048,466,12,3160,193.1,6.8,64.2,34.5%,1.5,68.2%
5,6,growlers,0.059,0.086,472,12,3095,182.3,6.6,55.5,36.0%,1.4,67.6%
6,7,hustle,0.058,0.060,449,12,3520,202.7,7.8,62.4,35.6%,2.0,66.8%
7,8,sol,0.055,0.060,484,12,3612,199.0,7.5,57.7,28.5%,1.9,61.8%
8,9,glory,0.049,0.045,441,12,3684,179.6,8.4,55.8,25.4%,2.4,60.1%
9,10,shred,0.048,0.035,481,12,3374,162.5,7.0,59.0,33.3%,1.8,64.0%


## Total aEC Sanity Check

Shown Space's `Tot-aEC` is a total, not an aEC/T rate. Sorting the all-possession summary by `total_aec` should line up with that table much more closely than the scoring-only views.

In [24]:
team_total_aec_low_to_high = team_aec_t_all_possessions.sort_values('total_aec', ascending=True).reset_index(drop=True).copy()
team_total_aec_low_to_high['rank'] = range(1, len(team_total_aec_low_to_high) + 1)
show_team_aec_t_table(team_total_aec_low_to_high)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,bighorns,-0.019,-0.114,541,12,2901,-53.9,5.4,48.1,36.6%,1.5,83.2%
1,2,havoc,0.008,-0.050,484,12,3485,26.2,7.2,54.1,38.6%,2.0,83.3%
2,3,thunderbirds,0.022,-0.039,458,11,3090,67.8,6.7,59.8,41.3%,1.9,81.2%
3,4,steel,0.023,-0.012,492,12,3131,71.7,6.4,56.3,38.8%,1.6,82.7%
4,5,union,0.025,-0.021,393,11,3297,83.1,8.4,62.1,33.8%,2.4,77.1%
5,6,phoenix,0.025,-0.023,484,12,3380,84.6,7.0,63.7,43.2%,1.7,81.0%
6,7,apex,0.033,-0.004,474,12,3146,104.5,6.6,63.3,37.1%,1.4,82.9%
7,8,royal,0.035,0.001,462,12,3105,108.6,6.7,55.4,32.3%,1.5,73.8%
8,9,rush,0.046,0.038,511,12,3123,142.2,6.1,58.7,39.5%,1.5,68.5%
9,10,cascades,0.043,0.035,482,12,3399,147.5,7.1,54.1,33.0%,1.7,63.1%


## All Scoring Possessions

This view only includes possessions that ended in goals. It answers a narrower question: when a team scores, how much aEC does that scoring possession create per throw? It should not be read as overall team aEC.

In [25]:
team_aec_t_all_scoring = summarize_team_aec_per_throw(
    league_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_scoring,
    title=f'{SEASON} UFA team aEC/T - all scoring possessions',
)

In [26]:
show_team_aec_t_table(team_aec_t_all_scoring)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,bighorns,0.145,0.275,156,12,1076,156.1,6.9,72.9,57.1%,1.8,86.5%
1,2,rush,0.142,0.260,256,12,1793,255.0,7.0,69.8,46.9%,1.7,68.0%
2,3,growlers,0.140,0.283,239,12,1713,240.1,7.2,65.7,43.9%,1.6,65.3%
3,4,steel,0.139,0.238,206,12,1497,207.9,7.3,73.2,53.4%,1.7,79.1%
4,5,alleycats,0.130,0.242,248,12,1910,247.7,7.7,72.6,49.6%,1.9,66.9%
5,6,cascades,0.126,0.238,234,12,1865,235.1,8.0,67.7,43.2%,1.8,67.9%
6,7,apex,0.126,0.203,211,12,1702,214.4,8.1,76.8,43.1%,1.7,80.1%
7,8,breeze,0.125,0.209,269,12,2140,267.8,8.0,73.9,32.7%,1.7,66.9%
8,9,shred,0.125,0.216,260,12,2094,260.8,8.1,73.1,39.6%,2.0,66.5%
9,10,windchill,0.123,0.221,273,12,2219,273.7,8.1,66.3,37.7%,2.3,59.7%


## Top 5 Non-Huck Scoring Possessions

Raw top-five aEC/T can overrate one-throw or very short scores. This version excludes hucks and requires at least `TOP_POSSESSION_MIN_THROWS` throws before selecting each team's top five.

In [27]:
non_huck_scoring_possessions = league_possessions[
    pd.to_numeric(league_possessions['huck_count'], errors='coerce').fillna(0).eq(0)
].copy()

team_top5_non_huck_aec_t = summarize_team_top_possessions(
    non_huck_scoring_possessions,
    top_n=TOP_POSSESSION_COUNT,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=TOP_POSSESSION_MIN_THROWS,
)

plot_team_metric_leaderboard(
    team_top5_non_huck_aec_t,
    metric='top_mean_metric',
    title=f'{SEASON} UFA top {TOP_POSSESSION_COUNT} non-huck scoring possession aEC/T, min {TOP_POSSESSION_MIN_THROWS} throws',
    x_label=f'top {TOP_POSSESSION_COUNT} mean aEC/T',
)

In [28]:
show_team_aec_t_table(team_top5_non_huck_aec_t, columns=TOP_POSSESSION_COLUMNS)

,rank,team_id,top_mean_metric,top_floor_metric,top_avg_throw_count,team_aec_per_throw,median_possession_aec_per_throw,possessions,throws
0,1,hustle,0.221,0.214,5.0,0.083,0.088,134,1644
1,2,empire,0.218,0.191,5.8,0.089,0.091,144,1639
2,3,royal,0.218,0.200,5.0,0.091,0.100,109,1199
3,4,windchill,0.216,0.210,5.0,0.091,0.100,130,1424
4,5,rush,0.214,0.201,5.0,0.093,0.101,90,965
5,6,alleycats,0.212,0.201,5.0,0.092,0.099,88,937
6,7,breeze,0.212,0.200,5.0,0.094,0.099,145,1542
7,8,flyers,0.211,0.200,5.0,0.084,0.093,146,1725
8,9,steel,0.209,0.190,5.0,0.093,0.103,69,758
9,10,growlers,0.208,0.200,5.0,0.087,0.108,90,1032


## Consistent Non-Huck Scoring aEC/T

This is still scoring-only, so turnovers are not included. It asks which teams repeatedly finish successful non-huck scores with high aEC/T. The next section adds turnovers back in.

In [29]:
eligible_non_huck_scoring = non_huck_scoring_possessions[
    pd.to_numeric(non_huck_scoring_possessions['throw_count'], errors='coerce').ge(CONSISTENCY_MIN_THROWS)
].copy()
non_huck_high_aec_t_threshold = pd.to_numeric(
    eligible_non_huck_scoring['aec_per_throw'],
    errors='coerce',
).quantile(0.75)

team_non_huck_aec_t_consistency = summarize_team_aec_consistency(
    non_huck_scoring_possessions,
    high_threshold=non_huck_high_aec_t_threshold,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
)

print(f'Eligible non-huck scoring possessions: {len(eligible_non_huck_scoring):,}')
print(f'High aEC/T threshold, 75th percentile: {non_huck_high_aec_t_threshold:.3f}')

plot_team_metric_leaderboard(
    team_non_huck_aec_t_consistency,
    metric='median_metric',
    title=f'{SEASON} UFA consistent non-huck scoring aEC/T, min {CONSISTENCY_MIN_THROWS} throws',
    x_label='median non-huck scoring aEC/T',
)

Eligible non-huck scoring possessions: 2,436
High aEC/T threshold, 75th percentile: 0.127


In [30]:
show_team_aec_t_table(team_non_huck_aec_t_consistency, columns=CONSISTENCY_COLUMNS)

,rank,team_id,median_metric,p75_metric,top_n_mean_metric,high_metric_share,team_aec_per_throw,possessions,throws,avg_throw_count,goal_share,turnover_share
0,1,growlers,0.108,0.131,0.204,30.0%,0.087,90,1032,11.5,100.0%,0.0%
1,2,bighorns,0.107,0.143,0.175,35.6%,0.093,45,479,10.6,100.0%,0.0%
2,3,steel,0.103,0.131,0.185,27.5%,0.093,69,758,11.0,100.0%,0.0%
3,4,spiders,0.102,0.132,0.198,27.3%,0.095,176,1884,10.7,100.0%,0.0%
4,5,rush,0.101,0.134,0.201,27.8%,0.093,90,965,10.7,100.0%,0.0%
5,6,apex,0.100,0.124,0.179,20.0%,0.093,100,1101,11.0,100.0%,0.0%
6,7,shred,0.100,0.134,0.198,29.0%,0.094,124,1333,10.8,100.0%,0.0%
7,8,royal,0.100,0.126,0.201,24.8%,0.091,109,1199,11.0,100.0%,0.0%
8,9,windchill,0.100,0.136,0.208,28.5%,0.091,130,1424,11.0,100.0%,0.0%
9,10,breeze,0.099,0.141,0.199,28.3%,0.094,145,1542,10.6,100.0%,0.0%


## Consistent Non-Huck aEC/T Including Turnovers

This uses all built possessions, filters out hucks, and includes turnovers. This is a better team-quality version of the consistency view because failed possessions count against the team.

In [31]:
non_huck_all_possessions = league_all_possessions[
    pd.to_numeric(league_all_possessions['huck_count'], errors='coerce').fillna(0).eq(0)
].copy()
eligible_non_huck_all = non_huck_all_possessions[
    pd.to_numeric(non_huck_all_possessions['throw_count'], errors='coerce').ge(CONSISTENCY_MIN_THROWS)
].copy()
non_huck_all_high_aec_t_threshold = pd.to_numeric(
    eligible_non_huck_all['aec_per_throw'],
    errors='coerce',
).quantile(0.75)

team_non_huck_all_aec_t_consistency = summarize_team_aec_consistency(
    non_huck_all_possessions,
    high_threshold=non_huck_all_high_aec_t_threshold,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
)

print(f'Eligible non-huck all possessions: {len(eligible_non_huck_all):,}')
print(f'High aEC/T threshold, 75th percentile: {non_huck_all_high_aec_t_threshold:.3f}')

plot_team_metric_leaderboard(
    team_non_huck_all_aec_t_consistency,
    metric='median_metric',
    title=f'{SEASON} UFA consistent non-huck aEC/T including turnovers, min {CONSISTENCY_MIN_THROWS} throws',
    x_label='median non-huck all-possession aEC/T',
)

Eligible non-huck all possessions: 4,235
High aEC/T threshold, 75th percentile: 0.101


In [32]:
show_team_aec_t_table(team_non_huck_all_aec_t_consistency, columns=CONSISTENCY_COLUMNS)

,rank,team_id,median_metric,p75_metric,top_n_mean_metric,high_metric_share,team_aec_per_throw,possessions,throws,avg_throw_count,goal_share,turnover_share
0,1,spiders,0.084,0.124,0.198,36.6%,0.064,246,2502,10.2,71.5%,27.2%
1,2,empire,0.082,0.111,0.198,30.5%,0.063,197,2110,10.7,73.1%,24.4%
2,3,breeze,0.079,0.121,0.199,32.7%,0.065,205,2046,10.0,70.7%,25.9%
3,4,windchill,0.079,0.112,0.208,32.0%,0.055,194,2034,10.5,67.0%,30.9%
4,5,flyers,0.076,0.119,0.204,33.5%,0.063,191,2105,11.0,76.4%,22.0%
5,6,sol,0.069,0.111,0.200,28.3%,0.052,226,2422,10.7,65.0%,32.3%
6,7,hustle,0.064,0.101,0.212,25.5%,0.047,204,2386,11.7,65.7%,32.4%
7,8,glory,0.064,0.091,0.183,16.7%,0.045,228,2698,11.8,64.9%,33.3%
8,9,shred,0.063,0.106,0.198,27.6%,0.043,217,2162,10.0,57.1%,41.5%
9,10,royal,0.060,0.106,0.201,26.7%,0.042,191,1975,10.3,57.1%,42.4%


## Default Possession-Shape Pool

This uses the same default filter as the top-possession shape work: O-line scoring possessions, long-field starts, hucks excluded.

In [33]:
analysis_possessions, analysis_paths = filter_analysis_possessions(
    league_possessions,
    league_paths,
)
team_aec_t_default_pool = summarize_team_aec_per_throw(
    analysis_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_default_pool,
    title=f'{SEASON} UFA team aEC/T - O-line non-huck long-field scores',
)

In [34]:
show_team_aec_t_table(team_aec_t_default_pool)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,spiders,0.095,0.110,97,12,1042,99.3,10.7,82.8,0.0%,2.1,100.0%
1,2,shred,0.094,0.108,78,12,847,80.0,10.9,87.3,0.0%,2.5,100.0%
2,3,royal,0.091,0.108,73,12,800,72.8,11.0,81.0,0.0%,2.4,100.0%
3,4,empire,0.090,0.109,91,12,1048,94.6,11.5,90.9,0.0%,2.6,100.0%
4,5,bighorns,0.090,0.111,34,11,371,33.4,10.9,85.1,0.0%,2.6,100.0%
5,6,breeze,0.090,0.106,96,12,1082,97.3,11.3,88.3,0.0%,2.5,100.0%
6,7,alleycats,0.089,0.104,45,12,513,45.9,11.4,88.7,0.0%,2.4,100.0%
7,8,phoenix,0.089,0.110,66,12,754,67.3,11.4,87.2,0.0%,2.5,100.0%
8,9,thunderbirds,0.088,0.105,60,11,670,58.8,11.2,89.2,0.0%,2.8,100.0%
9,10,apex,0.088,0.099,65,12,757,66.4,11.6,88.8,0.0%,2.5,100.0%
